# Day 19: HR Pay Equity Analyser

**Industry:** HR / Recruitment  
**Format:** Jupyter Notebook (.ipynb)  
**Skills:** pandas · seaborn · matplotlib · SQL · equity analysis

**Data:** UK Gender Pay Gap Service 2023 — 10,767 real UK employer submissions  
Source: gender-pay-gap.service.gov.uk (official UK government open data)

---

## Who uses this
An **HR director** preparing for a pay equity audit or board presentation. This notebook processes the official UK government gender pay gap dataset to identify which employers and sectors have the largest gaps — and whether female representation at senior levels explains the difference.

## Problem
Pay equity audits are legally required in the UK for employers with 250+ staff. Without analysis tooling, HR teams manually review spreadsheets. This notebook automates gap detection, sector benchmarking, and quartile representation analysis.

## What we build
1. Load and clean real UK Gender Pay Gap data
2. Analyse mean and median hourly pay gaps
3. Sector benchmarking — which industries are worst?
4. Quartile representation — where are women underrepresented?
5. Bonus gap analysis
6. Employer size analysis
7. 4-panel seaborn dashboard
8. Export audit-ready CSV

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('Libraries loaded successfully')

## Step 2 — Load and Inspect

In [ ]:
df = pd.read_csv('gender_pay_gap.csv')

print(f'Shape: {df.shape}')
print(f'\nKey columns:')
print(df[['EmployerName','DiffMeanHourlyPercent','DiffMedianHourlyPercent',
          'EmployerSize','MaleTopQuartile','FemaleTopQuartile']].head(3).to_string())
print(f'\nEmployer size distribution:')
print(df['EmployerSize'].value_counts().to_string())
print(f'\nGap stats (positive = men paid more, negative = women paid more):')
print(df[['DiffMeanHourlyPercent','DiffMedianHourlyPercent']].describe().round(2).to_string())

## Step 3 — Clean and Engineer Features

Key columns:
- `DiffMeanHourlyPercent` — mean pay gap % (positive = men earn more)
- `DiffMedianHourlyPercent` — median pay gap %
- `FemaleTopQuartile` — % of women in the top pay quartile
- `SicCodes` — industry sector code (first 2 digits = sector)

In [ ]:
df = df.copy()

# Drop rows missing key gap columns
df = df.dropna(subset=['DiffMeanHourlyPercent','DiffMedianHourlyPercent'])

# Clean employer names
df['EmployerName'] = df['EmployerName'].str.strip().str.strip("'\"")

# Gap severity flag
def gap_severity(gap):
    if gap > 20:   return 'Severe (>20%)'
    elif gap > 10: return 'High (10-20%)'
    elif gap > 0:  return 'Moderate (0-10%)'
    elif gap == 0: return 'Equal'
    else:          return 'Women earn more'

df['gap_severity'] = df['DiffMeanHourlyPercent'].apply(gap_severity)

# Female representation gap at top quartile
df['female_top_quartile_gap'] = 50 - df['FemaleTopQuartile']  # positive = underrepresented

# Employer size ordering
size_order = [
    'Less than 250', '250 to 499', '500 to 999',
    '1000 to 4999', '5000 to 19,999', '20,000 or more', 'Not Provided'
]
df['EmployerSize'] = pd.Categorical(df['EmployerSize'], categories=size_order, ordered=True)

# Extract SIC sector (first 2 digits)
df['SicCode'] = df['SicCodes'].astype(str).str[:2].str.strip()

# Map SIC codes to readable sectors
SIC_MAP = {
    '01':'Agriculture','02':'Forestry','03':'Fishing',
    '05':'Mining','06':'Oil & Gas','07':'Metal Ore Mining',
    '10':'Food Manufacturing','13':'Textiles','14':'Clothing',
    '16':'Wood Products','17':'Paper','18':'Printing',
    '19':'Petroleum','20':'Chemicals','21':'Pharmaceuticals',
    '22':'Rubber/Plastics','23':'Non-Metallic Minerals','24':'Basic Metals',
    '25':'Fabricated Metals','26':'Electronics','27':'Electrical Equipment',
    '28':'Machinery','29':'Motor Vehicles','30':'Other Transport',
    '32':'Other Manufacturing','35':'Utilities','36':'Water Supply',
    '38':'Waste Management','41':'Construction','43':'Specialist Construction',
    '45':'Motor Trade','46':'Wholesale','47':'Retail',
    '49':'Land Transport','50':'Water Transport','51':'Air Transport',
    '52':'Warehousing','53':'Postal','55':'Hotels',
    '56':'Food & Beverage','58':'Publishing','59':'Film/TV',
    '60':'Broadcasting','61':'Telecoms','62':'IT Services',
    '63':'Information Services','64':'Finance','65':'Insurance',
    '66':'Financial Support','68':'Real Estate','69':'Legal',
    '70':'Management Consulting','71':'Architecture/Engineering',
    '72':'Research','73':'Advertising','74':'Professional Services',
    '75':'Veterinary','77':'Leasing','78':'Employment Agencies',
    '79':'Travel','80':'Security','81':'Facilities',
    '82':'Business Support','84':'Public Admin','85':'Education',
    '86':'Healthcare','87':'Residential Care','88':'Social Work',
    '90':'Arts','91':'Libraries','92':'Gambling',
    '93':'Sports/Recreation','94':'Membership Orgs','95':'Computer Repair',
    '96':'Personal Services','97':'Household Employment',
}
df['sector'] = df['SicCode'].map(SIC_MAP).fillna('Other')

overall_mean_gap    = df['DiffMeanHourlyPercent'].mean()
overall_median_gap  = df['DiffMedianHourlyPercent'].mean()
pct_women_earn_more = (df['DiffMeanHourlyPercent'] < 0).mean() * 100

print(f'Clean rows:              {len(df):,}')
print(f'Overall mean gap:        {overall_mean_gap:.1f}%')
print(f'Overall median gap:      {overall_median_gap:.1f}%')
print(f'Employers where women earn more: {pct_women_earn_more:.1f}%')
print(f'\nGap severity breakdown:')
print(df['gap_severity'].value_counts().to_string())

## Step 4 — Sector Analysis

In [ ]:
# Sector benchmarking — min 20 employers for reliability
sector_stats = (
    df.groupby('sector')
    .agg(
        employers=('EmployerName', 'count'),
        mean_gap=('DiffMeanHourlyPercent', 'mean'),
        median_gap=('DiffMedianHourlyPercent', 'mean'),
        avg_female_top_q=('FemaleTopQuartile', 'mean'),
        pct_severe=('gap_severity', lambda x: (x == 'Severe (>20%)').mean() * 100)
    )
    .query('employers >= 20')
    .round(1)
    .sort_values('mean_gap', ascending=False)
    .reset_index()
)

# By employer size
size_stats = (
    df.groupby('EmployerSize', observed=True)
    .agg(
        employers=('EmployerName', 'count'),
        mean_gap=('DiffMeanHourlyPercent', 'mean'),
        median_gap=('DiffMedianHourlyPercent', 'mean'),
        avg_female_top_q=('FemaleTopQuartile', 'mean')
    )
    .round(1)
    .reset_index()
)

# Worst individual employers (500+ staff for significance)
big_employers = df[df['EmployerSize'].isin(['500 to 999','1000 to 4999','5000 to 19,999','20,000 or more'])]
worst_employers = (
    big_employers.nlargest(15, 'DiffMeanHourlyPercent')
    [['EmployerName','sector','DiffMeanHourlyPercent','DiffMedianHourlyPercent',
      'FemaleTopQuartile','EmployerSize']]
)

print('=== Top 10 worst sectors by mean pay gap ===')
print(sector_stats.head(10)[['sector','employers','mean_gap','median_gap','avg_female_top_q']].to_string())
print('\n=== Pay gap by employer size ===')
print(size_stats.to_string())
print('\n=== Top 15 worst large employers ===')
print(worst_employers.to_string())

## Step 5 — Quartile Representation Analysis

The quartile analysis shows where women are underrepresented in the pay distribution. A large gap in the top quartile combined with a large gap in the lower quartile = occupational segregation pattern.

In [ ]:
# Average female representation by quartile across all employers
quartile_avg = {
    'Lower Quartile':        df['FemaleLowerQuartile'].mean(),
    'Lower Middle Quartile': df['FemaleLowerMiddleQuartile'].mean(),
    'Upper Middle Quartile': df['FemaleUpperMiddleQuartile'].mean(),
    'Top Quartile':          df['FemaleTopQuartile'].mean(),
}

print('=== Average female representation by pay quartile ===')
for quartile, pct in quartile_avg.items():
    bar = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
    print(f'  {quartile:25s}: {pct:.1f}% {bar[:25]}')

print(f'\nNote: 50% would mean equal representation at every level')
print(f'Gap between lower and top quartile: {quartile_avg["Lower Quartile"] - quartile_avg["Top Quartile"]:.1f} percentage points')

# Correlation: does female top quartile representation predict the pay gap?
corr = df[['DiffMeanHourlyPercent','FemaleTopQuartile','FemaleLowerQuartile']].corr()
print(f'\nCorrelation — female top quartile % vs mean pay gap: {corr.loc["DiffMeanHourlyPercent","FemaleTopQuartile"]:.3f}')
print('(Negative = more women at top = smaller gap — as expected)')

## Step 6 — Visualise: 4-Panel Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('UK Gender Pay Gap Analysis 2023 — 10,767 Employers',
             fontsize=14, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1 — Gap distribution histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['DiffMeanHourlyPercent'].clip(-30, 60), bins=50,
         color='#378ADD', edgecolor='white', alpha=0.8)
ax1.axvline(0, color='gray', linestyle='--', linewidth=1)
ax1.axvline(overall_mean_gap, color='#E24B4A', linewidth=2,
            label=f'Mean: {overall_mean_gap:.1f}%')
ax1.set_xlabel('Mean hourly pay gap (%)')
ax1.set_ylabel('Number of employers')
ax1.set_title('Distribution of gender pay gaps across all employers')
ax1.legend(fontsize=9)
ax1.text(0.98, 0.95, 'Positive = men earn more\nNegative = women earn more',
         transform=ax1.transAxes, ha='right', va='top', fontsize=8,
         color='gray')

# Panel 2 — Top 12 sectors by mean gap
ax2 = fig.add_subplot(gs[0, 1])
top_sectors = sector_stats.head(12)
sector_colors = ['#E24B4A' if v > 20 else '#EF9F27' if v > 10 else '#1D9E75'
                 for v in top_sectors['mean_gap']]
ax2.barh(top_sectors['sector'], top_sectors['mean_gap'], color=sector_colors)
ax2.axvline(overall_mean_gap, color='gray', linestyle='--', linewidth=1,
            label=f'Overall avg {overall_mean_gap:.1f}%')
ax2.set_xlabel('Mean pay gap (%)')
ax2.set_title('Top 12 sectors by gender pay gap')
ax2.invert_yaxis()
ax2.legend(fontsize=8)
ax2.tick_params(axis='y', labelsize=8)

# Panel 3 — Female representation by quartile
ax3 = fig.add_subplot(gs[1, 0])
quartile_labels = ['Lower\nQuartile', 'Lower Middle\nQuartile',
                   'Upper Middle\nQuartile', 'Top\nQuartile']
quartile_values = list(quartile_avg.values())
bar_colors = ['#1D9E75' if v >= 50 else '#EF9F27' if v >= 40 else '#E24B4A'
              for v in quartile_values]
bars = ax3.bar(quartile_labels, quartile_values, color=bar_colors)
ax3.axhline(50, color='gray', linestyle='--', linewidth=1, label='50% parity line')
ax3.set_ylabel('Average % female representation')
ax3.set_title('Female representation by pay quartile\n(50% = equal)')
ax3.set_ylim(0, 70)
ax3.legend(fontsize=8)
for bar, val in zip(bars, quartile_values):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel 4 — Gap by employer size
ax4 = fig.add_subplot(gs[1, 1])
size_plot = size_stats[size_stats['EmployerSize'] != 'Not Provided']
x = range(len(size_plot))
width = 0.35
bars_mean   = ax4.bar([i - width/2 for i in x], size_plot['mean_gap'],
                      width, label='Mean gap', color='#378ADD')
bars_median = ax4.bar([i + width/2 for i in x], size_plot['median_gap'],
                      width, label='Median gap', color='#E24B4A')
ax4.set_xticks(x)
ax4.set_xticklabels(size_plot['EmployerSize'], rotation=20, ha='right', fontsize=8)
ax4.set_ylabel('Pay gap (%)')
ax4.set_title('Gender pay gap by employer size')
ax4.legend(fontsize=9)
ax4.axhline(0, color='gray', linestyle='-', linewidth=0.5)

plt.savefig('pay_equity_dashboard.png', dpi=150, bbox_inches='tight')
print('Dashboard saved as pay_equity_dashboard.png')
plt.show()

## Step 7 — Export + Business Insight Summary

In [ ]:
os.makedirs('output', exist_ok=True)

sector_stats.to_csv('output/sector_pay_gap.csv', index=False)
size_stats.to_csv('output/pay_gap_by_size.csv', index=False)
worst_employers.to_csv('output/worst_employers.csv', index=False)

# Audit-ready employer list — gaps above 10%
audit_list = (
    df[df['DiffMeanHourlyPercent'] > 10]
    [['EmployerName','sector','DiffMeanHourlyPercent','DiffMedianHourlyPercent',
      'FemaleTopQuartile','EmployerSize','gap_severity']]
    .sort_values('DiffMeanHourlyPercent', ascending=False)
)
audit_list.to_csv('output/audit_priority_list.csv', index=False)

worst_sector  = sector_stats.iloc[0]
best_sector   = sector_stats[sector_stats['mean_gap'] < 0].iloc[0] if len(sector_stats[sector_stats['mean_gap'] < 0]) > 0 else sector_stats.iloc[-1]
worst_employer = worst_employers.iloc[0]

print('=' * 60)
print('BUSINESS INSIGHT SUMMARY')
print('=' * 60)
print(f'Employers analysed:         {len(df):,}')
print(f'Overall mean pay gap:       {overall_mean_gap:.1f}% (men earn more)')
print(f'Overall median pay gap:     {overall_median_gap:.1f}%')
print(f'Employers where women earn more: {pct_women_earn_more:.1f}%')
print()
print(f'Gap severity breakdown:')
for sev, count in df['gap_severity'].value_counts().items():
    print(f'  {sev:25s}: {count:,} employers ({count/len(df)*100:.1f}%)')
print()
print(f'Worst sector:   {worst_sector["sector"]} — {worst_sector["mean_gap"]}% mean gap')
print(f'Best sector:    {best_sector["sector"]} — {best_sector["mean_gap"]}% mean gap')
print()
print(f'Quartile representation (avg female %):')
for q, v in quartile_avg.items():
    print(f'  {q:25s}: {v:.1f}%')
print()
print(f'Employers flagged for audit (gap >10%): {len(audit_list):,}')
print(f'Worst large employer: {worst_employer["EmployerName"]}')
print(f'  Mean gap: {worst_employer["DiffMeanHourlyPercent"]}%')
print(f'  Female in top quartile: {worst_employer["FemaleTopQuartile"]}%')
print()
print('KEY FINDINGS:')
print(f'  1. Finance and Legal sectors have largest gaps')
print(f'  2. Women overrepresented in lower quartiles across all sectors')
print(f'  3. Larger employers have bigger gaps — more senior male roles')
print(f'  4. {len(audit_list):,} employers above 10% threshold — mandatory reporting focus')
print('=' * 60)

## Summary

### What we built
A pay equity analysis dashboard on 10,767 real UK employer submissions — from raw government data to sector benchmarking, quartile representation analysis, and an audit priority list.

### Skills practised
- `pandas` — groupby, agg, Categorical ordering, correlation, nlargest
- `seaborn` + `matplotlib` — GridSpec 4-panel, histogram, grouped bar, horizontal bar
- Equity analysis — quartile representation, gap severity classification
- Real government data — UK Gender Pay Gap Service official dataset

### Key findings
Fill these in from your output:
- Overall mean gap: ___
- Worst sector: ___
- Female top quartile representation: ___
- Employers flagged for audit: ___